# Joint fine-tuning of DistilBERT with structured campaign features

This notebook documents **binary classification** of crowdfunding campaign outcomes (**successful** versus **failed**). The text modality is encoded with **DistilBERT** (`distilbert-base-uncased`), while a fixed **tabular** vector (category dummies, goal-related fields, duration, sentiment, readability, and related engineered features) is concatenated with a **mean-pooled** representation of the token sequence before a small feed-forward **classifier head**. Parameters of the text encoder and the head are updated **jointly** under the Hugging Face `Trainer` API.

Training and validation rows are read from `data/features/train.csv` and `data/features/val.csv` as provided; the notebook does not construct a new random split.


In [17]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    TrainingArguments,
    Trainer
)

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
import evaluate
import os

## Tabular feature specification

Predictor names for the structured block are read from `data/features/features_no_scale.txt` (binary or already normalized columns) and `data/features/features_scale.txt` (numeric columns to be **z-scored**). The following cell reports the counts (21 no-scale and 8 scale columns in the current configuration, for **29** structured dimensions after concatenation).


In [18]:
def load_feature_names(path):
    with open(path) as f:
        return [line.strip() for line in f if line.strip()]

FEATURES_NO_SCALE = load_feature_names("data/features/features_no_scale.txt")
FEATURES_SCALE = load_feature_names("data/features/features_scale.txt")

print("No-scale features:", len(FEATURES_NO_SCALE))
print("Scale features:", len(FEATURES_SCALE))

No-scale features: 21
Scale features: 8


## Raw table load

The training and validation feature tables are read from `data/features/train.csv` and `data/features/val.csv`. The following cell prints row and column counts for the raw files before filtering.


In [19]:
train_raw = pd.read_csv("data/features/train.csv")
val_raw = pd.read_csv("data/features/val.csv")

print("Train raw shape:", train_raw.shape)
print("Val raw shape:", val_raw.shape)

Train raw shape: (66096, 42)
Val raw shape: (22032, 42)


## Filtering, text field, and labels

The next cell restricts rows to terminal outcomes **successful** or **failed**, concatenates **name** and **blurb** into a single **text** column, defines the binary **label**, drops rows with missing text, label, or any required structured field, and retains only the columns needed for modeling. A small preview of the prepared training frame is shown.


In [20]:
def prepare_aligned(df):
    df = df[df["state"].isin(["successful", "failed"])].copy()
    df["text"] = df["name"].fillna("") + " " + df["blurb"].fillna("")
    df["label"] = (df["state"] == "successful").astype(int)

    needed_cols = ["text", "label"] + FEATURES_NO_SCALE + FEATURES_SCALE
    df = df.dropna(subset=["text", "label"]).copy()

    # keep only rows where structured features exist
    df = df.dropna(subset=FEATURES_NO_SCALE + FEATURES_SCALE).reset_index(drop=True)

    return df[needed_cols]

train_df = prepare_aligned(train_raw)
val_df = prepare_aligned(val_raw)

print("Train prepared shape:", train_df.shape)
print("Val prepared shape:", val_df.shape)
train_df.head()

Train prepared shape: (66096, 31)
Val prepared shape: (22032, 31)


,text,label,cat_Art,cat_Comics,cat_Crafts,cat_Dance,cat_Design,cat_Fashion,cat_Film & Video,cat_Food,...,country_US,z-score_log_goal,duration,CCI_index,blurb_length,sentiment_score,readability_score,name_blurb_similarity,log_goal,CCI_per_goal
0,Bring Art ala Carte back to the community! Art...,0,1,0,0,0,0,0,0,0,...,1,1.150449,30,101.23980,21,0.4738,71.291786,0.906809,10.126671,0.004050
1,The PRESENT MIND Deck Mentalist Grant Price de...,1,0,0,0,0,0,0,0,0,...,1,0.750864,28,98.42246,22,0.7650,75.320357,0.770244,9.105091,0.010936
2,Zeus-X Pro ⚡ - The World's Most Futuristic Uni...,1,0,0,0,0,0,0,0,0,...,0,-0.599290,35,99.30466,22,0.6221,76.460909,0.737919,8.382949,0.022720
3,Make 100 : The Spotting Chart Project - WWII A...,1,0,0,0,0,1,0,0,0,...,1,-0.649362,21,101.38930,20,0.3612,59.635000,0.720486,6.685861,0.126737
4,Art & Hue | Graphic Pop Art | Bespoke & Custom...,1,1,0,0,0,0,0,0,0,...,0,0.641695,30,101.81250,24,0.2500,66.423370,0.789146,8.353842,0.023981


## Structured numeric matrices

**StandardScaler** is fit on the training partition’s scale columns only, then applied to validation. No-scale columns are cast to float and stacked with the scaled block. **nan_to_num** replaces non-finite values so the resulting **29**-dimensional vectors can be aligned with tokenized examples in a later step.


In [21]:
scaler = StandardScaler()

train_scaled = scaler.fit_transform(train_df[FEATURES_SCALE].astype(np.float32))
val_scaled = scaler.transform(val_df[FEATURES_SCALE].astype(np.float32))

train_no_scale = train_df[FEATURES_NO_SCALE].astype(np.float32).to_numpy()
val_no_scale = val_df[FEATURES_NO_SCALE].astype(np.float32).to_numpy()

X_train_struct = np.hstack([train_no_scale, train_scaled]).astype(np.float32)
X_val_struct = np.hstack([val_no_scale, val_scaled]).astype(np.float32)

X_train_struct = np.nan_to_num(X_train_struct, nan=0.0, posinf=0.0, neginf=0.0)
X_val_struct = np.nan_to_num(X_val_struct, nan=0.0, posinf=0.0, neginf=0.0)

print("Structured train shape:", X_train_struct.shape)
print("Structured val shape:", X_val_struct.shape)

Structured train shape: (66096, 29)
Structured val shape: (22032, 29)


## Hugging Face `Dataset` construction

Lightweight tables containing **text** and **label** are converted with `Dataset.from_pandas` for compatibility with `map` and the `Trainer`. Structured features remain aligned row-for-row with these examples and are attached in a later step.


In [22]:
train_text_df = pd.DataFrame({
    "text": train_df["text"].values,
    "label": train_df["label"].values
})

val_text_df = pd.DataFrame({
    "text": val_df["text"].values,
    "label": val_df["label"].values
})

train_dataset = Dataset.from_pandas(train_text_df)
val_dataset = Dataset.from_pandas(val_text_df)

## Tokenizer

The same pretrained checkpoint name is used for the **AutoTokenizer** as for the text encoder below, keeping subword vocabulary and model weights consistent.


In [23]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

## Text tokenization

Each campaign string is tokenized with **truncation**, **padding to `max_length=128`**, and fixed maximum length so batches are rectangular. The `map` operation adds `input_ids` and `attention_mask` to every split.


In [24]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/66096 [00:00<?, ? examples/s]

Map:   0%|          | 0/22032 [00:00<?, ? examples/s]

## Joining structured features with tokenized examples

The aligned **29**-dimensional vectors are appended as a list column **structured_features** on both train and validation datasets so each example carries its tabular context into the collated batch.


In [25]:
train_dataset = train_dataset.add_column("structured_features", X_train_struct.tolist())
val_dataset = val_dataset.add_column("structured_features", X_val_struct.tolist())

## Tensor column layout for training

Plain **text** strings are removed after tokenization to avoid passing redundant fields to the model. **set_format** restricts each example to **input_ids**, **attention_mask**, **structured_features**, and **label** as PyTorch tensors for the dataloader.


In [26]:
train_dataset = train_dataset.remove_columns(["text"])
val_dataset = val_dataset.remove_columns(["text"])

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "structured_features", "label"]
)

val_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "structured_features", "label"]
)

print(train_dataset[0].keys())
print(train_dataset[0]["structured_features"].shape)

dict_keys(['label', 'input_ids', 'attention_mask', 'structured_features'])
torch.Size([29])


## Metrics logged each evaluation epoch

The `compute_metrics` callback reports **accuracy**, **F1**, **precision**, **recall**, and **ROC AUC** on the validation split. Class probabilities for AUC are taken as the softmax of the logits for the **positive** (successful) class.


In [27]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    probs_pos = torch.softmax(torch.from_numpy(logits), dim=-1).numpy()[:, 1]
    roc_auc = roc_auc_score(labels, probs_pos)

    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels)
    precision = precision_metric.compute(predictions=predictions, references=labels)
    recall = recall_metric.compute(predictions=predictions, references=labels)

    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"],
        "precision": precision["precision"],
        "recall": recall["recall"],
        "roc_auc": float(roc_auc),
    }

## Architecture: `DistilBERTWithStructuredFeatures`

The module wraps **AutoModel** (DistilBERT) and applies **mean pooling** over the last hidden state using the attention mask. The pooled **768**-dimensional vector is concatenated with **structured_features**, passed through **dropout**, a **256**-unit ReLU layer with dropout, and a **2**-way linear output. **CrossEntropyLoss** is applied when `labels` are supplied; the forward pass returns **loss** and **logits** for use with `Trainer`.


In [28]:
class DistilBERTWithStructuredFeatures(nn.Module):
    def __init__(self, model_name, structured_dim, num_labels=2, dropout=0.2, hidden_dim=256):
        super().__init__()

        self.text_encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)

        text_hidden_size = self.text_encoder.config.hidden_size  # 768 for DistilBERT

        self.classifier = nn.Sequential(
            nn.Linear(text_hidden_size + structured_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_labels)
        )

        self.loss_fn = nn.CrossEntropyLoss()

    def mean_pooling(self, last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        masked_embeddings = last_hidden_state * mask
        summed = masked_embeddings.sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        return summed / counts

    def forward(self, input_ids=None, attention_mask=None, structured_features=None, labels=None):
        outputs = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        last_hidden_state = outputs.last_hidden_state
        pooled_text = self.mean_pooling(last_hidden_state, attention_mask)
        pooled_text = self.dropout(pooled_text)

        combined = torch.cat([pooled_text, structured_features], dim=1)
        logits = self.classifier(combined)

        loss = None
        if labels is not None:
            loss = self.loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}

## Model instantiation

The classifier input dimension is **768 + 29 = 797**. Dropout and hidden width follow the hyperparameters in the class definition above; the text encoder starts from pretrained DistilBERT weights and is fine-tuned together with the head.


In [29]:
structured_dim = X_train_struct.shape[1]

model = DistilBERTWithStructuredFeatures(
    model_name=model_name,
    structured_dim=structured_dim,
    num_labels=2,
    dropout=0.2,
    hidden_dim=256
)

print("Structured dim:", structured_dim)

Structured dim: 29


## Training arguments

Checkpoints and logs are written under `./joint_distilbert_structured_results`. Optimization uses **2** epochs, batch size **8** per device for train and eval, learning rate **2e-5**, weight decay **0.01**, and **epoch-level** evaluation, logging, and saving. The run with the best validation **F1** is retained at the end (`load_best_model_at_end`).


In [30]:
training_args = TrainingArguments(
    output_dir="./joint_distilbert_structured_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=1,
    report_to="none"
)

## `Trainer` setup

The Hugging Face **Trainer** receives the custom `nn.Module`, training and validation datasets, tokenizer (for decoding utilities and saving), and the metric function defined earlier. Default data collation stacks tensor fields into batches compatible with the model’s `forward` signature.


In [31]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/var/folders/0f/1mt731m54b51gldjgy_zszqr0000gn/T/ipykernel_12771/2721217396.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


## Checkpoint loading versus end-to-end training

Saved weights under `joint_distilbert_structured_results/checkpoint-16524` match the architecture instantiated above. The following code cell can load **`model.safetensors`** (or **`pytorch_model.bin`**) into the in-memory `model` attached to the `Trainer`, which allows the subsequent **evaluation** and **predict** cells to run without repeating the full fine-tuning schedule. Setting **`RUN_TRAINING = True`** and clearing or changing **`CHECKPOINT_DIR`** restores the standard path where **`trainer.train()`** optimizes parameters from the current initialization (or from loaded weights if a directory is still provided before training).


In [32]:
# Same folder as TrainingArguments output_dir; adjust if the notebook is run from another cwd.
CHECKPOINT_DIR = "./joint_distilbert_structured_results/checkpoint-16524"
# Set True to run full training instead of relying on a saved checkpoint.
RUN_TRAINING = False

_loaded_from_checkpoint = False
if CHECKPOINT_DIR and os.path.isdir(CHECKPOINT_DIR):
    weights_safetensors = os.path.join(CHECKPOINT_DIR, "model.safetensors")
    weights_bin = os.path.join(CHECKPOINT_DIR, "pytorch_model.bin")
    if os.path.isfile(weights_safetensors):
        from safetensors.torch import load_file

        state = load_file(weights_safetensors)
    elif os.path.isfile(weights_bin):
        state = torch.load(weights_bin, map_location="cpu", weights_only=True)
    else:
        raise FileNotFoundError(
            f"No model.safetensors or pytorch_model.bin in {CHECKPOINT_DIR!r}"
        )
    model.load_state_dict(state, strict=True)
    _loaded_from_checkpoint = True
    print("Loaded weights into `model` from:", CHECKPOINT_DIR)
else:
    if not RUN_TRAINING:
        print(
            "CHECKPOINT_DIR is missing or not a directory; set RUN_TRAINING = True "
            "or fix CHECKPOINT_DIR before evaluating."
        )


Loaded weights into `model` from: ./joint_distilbert_structured_results/checkpoint-16524


## Training loop execution

When **`RUN_TRAINING`** is **True**, **`trainer.train()`** runs the configured schedule. Otherwise training is skipped in favor of weights already loaded from disk or the fresh pretrained initialization, and a short status line records which case applies.


In [33]:
if RUN_TRAINING:
    trainer.train()
else:
    # Full training loop (optional). Uncomment the next line to train regardless of RUN_TRAINING.
    # trainer.train()
    if _loaded_from_checkpoint:
        print("Skipped trainer.train(); using weights loaded from checkpoint.")
    else:
        print(
            "Skipped trainer.train(); model is still at initialization weights "
            "(no valid CHECKPOINT_DIR and RUN_TRAINING is False)."
        )


Skipped trainer.train(); using weights loaded from checkpoint.


## Validation metrics after training or loading

**`trainer.evaluate()`** aggregates loss and all metrics returned by `compute_metrics` on the validation dataset. This cell is suitable for reporting headline numbers in a results table alongside other modeling approaches.


In [34]:
val_results = trainer.evaluate()
val_results

{'eval_loss': 0.5063025951385498,
 'eval_model_preparation_time': 0.0006,
 'eval_accuracy': 0.7824527959331881,
 'eval_f1': 0.8236765625574808,
 'eval_precision': 0.7931278781438187,
 'eval_recall': 0.8566727884909703,
 'eval_roc_auc': 0.8570125056905512,
 'eval_runtime': 100.1208,
 'eval_samples_per_second': 220.054,
 'eval_steps_per_second': 27.507}

## Held-out predictions and ROC AUC

**`trainer.predict`** emits logits for every validation example; argmax yields class predictions, and a secondary line prints **ROC AUC** using softmax probabilities for class **1**, which is useful when comparing ranking quality to threshold-free metrics such as accuracy alone.


In [35]:
pred_output = trainer.predict(val_dataset)
logits_val_pred = pred_output.predictions
preds = np.argmax(logits_val_pred, axis=-1)
val_probs = torch.softmax(torch.from_numpy(logits_val_pred), dim=-1).numpy()[:, 1]
print(
    "ROC AUC (val predict):",
    float(roc_auc_score(pred_output.label_ids, val_probs)),
)
print(preds[:10])

ROC AUC (val predict): 0.8570125056905512
[1 0 0 1 1 1 1 1 1 0]


## Validation examples by category and predicted success probability

For each of **Film & Video**, **Publishing**, **Art**, **Music**, and **Technology**, the table lists validation rows with that category flag (**`cat_*` = 1**): **two** campaigns with the **lowest** softmax probability of class **1** (successful) and **two** with the **highest**. Probabilities reuse the **validation** **`trainer.predict`** output when **`pred_output`** is already in memory. Blurbs are aligned from **`val_raw`** with the **same filter** as `prepare_aligned`, so the data-prep cell does not need to be re-run for blurbs.


In [39]:
def _val_blurbs_same_order_as_val_df(raw_df):
    """Same filtering as `prepare_aligned`; returns blurb text in validation row order."""
    df = raw_df[raw_df["state"].isin(["successful", "failed"])].copy()
    df["text"] = df["name"].fillna("") + " " + df["blurb"].fillna("")
    df["label"] = (df["state"] == "successful").astype(int)
    df = df.dropna(subset=["text", "label"]).copy()
    df = df.dropna(subset=FEATURES_NO_SCALE + FEATURES_SCALE).reset_index(drop=True)
    return df["blurb"].fillna("").astype(str).to_numpy()


try:
    _po = pred_output
    logits_tab = _po.predictions
    if logits_tab.shape[0] != len(val_df):
        raise ValueError("pred_output length mismatch")
except (NameError, ValueError, AttributeError):
    _po = trainer.predict(val_dataset)
    logits_tab = _po.predictions

probs_success = torch.softmax(torch.from_numpy(logits_tab), dim=-1).numpy()[:, 1]
val_blurbs = _val_blurbs_same_order_as_val_df(val_raw)
y_true = val_df["label"].to_numpy()

# (display name, one-hot column in `val_df` — matches `features_no_scale.txt`)
CATEGORY_ROWS = [
    ("Film & Video", "cat_Film & Video"),
    ("Publishing", "cat_Publishing"),
    ("Art", "cat_Art"),
    ("Music", "cat_Music"),
    ("Technology", "cat_Technology"),
]

parts = []
for display_name, col in CATEGORY_ROWS:
    if col not in val_df.columns:
        raise KeyError(f"Missing column {col!r} in val_df")
    mask = val_df[col].fillna(0).astype(int).to_numpy() == 1
    if mask.sum() == 0:
        continue
    sub = pd.DataFrame(
        {
            "category": display_name,
            "blurb": val_blurbs[mask],
            "prob_success": probs_success[mask],
            "true_outcome": np.where(y_true[mask] == 1, "successful", "failed"),
        }
    )
    low2 = sub.nsmallest(2, "prob_success").copy()
    low2["set"] = "Lowest P(success)"
    high2 = sub.nlargest(2, "prob_success").copy()
    high2["set"] = "Highest P(success)"
    parts.append(pd.concat([low2, high2], ignore_index=True))

extremes_by_cat = pd.concat(parts, ignore_index=True)
extremes_by_cat = extremes_by_cat[["category", "set", "prob_success", "true_outcome", "blurb"]]
cat_order = [name for name, _ in CATEGORY_ROWS]
extremes_by_cat["category"] = pd.Categorical(
    extremes_by_cat["category"], categories=cat_order, ordered=True
)
extremes_by_cat["set"] = pd.Categorical(
    extremes_by_cat["set"],
    categories=["Lowest P(success)", "Highest P(success)"],
    ordered=True,
)
extremes_by_cat = extremes_by_cat.sort_values(["category", "set"])
pd.set_option("display.max_colwidth", None)
extremes_by_cat


,category,set,prob_success,true_outcome,blurb
0,Film & Video,Lowest P(success),0.001137,failed,"This cartoon is about the main character ""PG"" Porky/Glenn who tells stories about his life experiences as a child and adult."
1,Film & Video,Lowest P(success),0.003256,failed,the story of in a movie of scam and pozie dealings of a man who was to think that america roads were paved with gold
2,Film & Video,Highest P(success),0.996592,successful,A post-apocalyptic love story with all practical FX. Winner of two Best Feature awards. Get your name in the credits and a signed copy!
3,Film & Video,Highest P(success),0.993606,successful,A narrative film following a woman’s journey from her home in San Francisco into her family’s past at the shores of the Salton Sea.
4,Publishing,Lowest P(success),0.003653,failed,Wanting to finish Publishing the last 3 of the book series. Also to have it adapted for Tv mini series.
5,Publishing,Lowest P(success),0.006651,failed,We aim to make high quality independent music accessible to everyone. Help support emerging artists by giving them a place to be heard
6,Publishing,Highest P(success),0.999025,successful,"A zine focused on the stories of magical girls (or boys, etc) fighting against social issues in our dystopian world."
7,Publishing,Highest P(success),0.998770,successful,A group of diverse creators bring quests for social change to the magical girl genre and resources to help you join the fight.
8,Art,Lowest P(success),0.002585,failed,Opening an art Studio with today's technology and current art students for tutoring to every person to acess for a small fee.
9,Art,Lowest P(success),0.003343,failed,My name is Mitchell and I make thick or thin framed canvases. I am looking for help to get equipment of my own such as a printer.
